In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load tumhari cleaned files
train_df = pd.read_csv("C:/Users/Z/plagiarism-detection/data/processed/quora_train.csv")
test_df = pd.read_csv("C:/Users/Z/plagiarism-detection/data/processed/quora_test.csv")

print(f"Train: {len(train_df)}, Test: {len(test_df)}")


# Pehle train data pe TF-IDF fit karo
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),        # Unigrams + bigrams
    stop_words='english',       # Remove common words
    sublinear_tf=True,          # Smooth term frequency
    min_df=3,                   # Ignore very rare words
    max_features=20000          # Keep top 20k features
) 

# Train ke dono columns ko combine karo
train_texts = train_df['q1_clean'].tolist() + train_df['q2_clean'].tolist()
vectorizer.fit(train_texts)

print("TF-IDF vocabulary size:", len(vectorizer.get_feature_names_out()))

# ========== FIND BEST THRESHOLD ==========
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score



similarities = []
for idx, row in test_df.iterrows():
    vec1 = vectorizer.transform([row['q1_clean']])
    vec2 = vectorizer.transform([row['q2_clean']])
    sim = cosine_similarity(vec1, vec2)[0][0]
    similarities.append(sim)


true_labels = test_df['is_duplicate'].tolist()

print("="*50)
print("FINDING BEST THRESHOLD")
print("="*50)

best_threshold = 0
best_f1 = 0

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    preds = [1 if sim > threshold else 0 for sim in similarities]
    f1 = f1_score(true_labels, preds)
    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds, zero_division=0)
    rec = recall_score(true_labels, preds, zero_division=0)
    
    print(f"Threshold {threshold}: Acc={acc:.4f}, P={prec:.4f}, R={rec:.4f}, F1={f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print("\n" + "="*50)
print(f"✅ BEST THRESHOLD: {best_threshold}")
print(f"✅ BEST F1-SCORE: {best_f1:.4f}")
print("="*50)


# Test data ke liye predictions
predictions = []

for idx, row in test_df.iterrows():
    # Dono questions ke vectors banao
    vec1 = vectorizer.transform([row['q1_clean']])
    vec2 = vectorizer.transform([row['q2_clean']])
    
    # Cosine similarity nikaalo
    similarity = cosine_similarity(vec1, vec2)[0][0]
    
    
    pred = 1 if similarity > 0.6 else 0
    predictions.append(pred)
    
# True labels
true_labels = test_df['is_duplicate'].tolist()



# Metrics calculate karo
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

print("="*40)
print("MODEL 1: TF-IDF + Cosine Similarity")
print("="*40)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")



import joblib
import os
import json

# Create folders if not exist
os.makedirs("trained_models", exist_ok=True)
os.makedirs("results", exist_ok=True)

# 1. Save the trained vectorizer (model)
joblib.dump(vectorizer, "trained_models/model1_vectorizer.pkl")
print("✅ Model saved to: trained_models/model1_vectorizer.pkl")

# 2. Save metrics (if you have them)

if 'accuracy' in locals():
    results = {
        'model': 'TF-IDF + Cosine Similarity',
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }
    
    with open("results/model1_results.json", 'w') as f:
        json.dump(results, f, indent=4)
    print("✅ Results saved to: results/model1_results.json")
else:
    print("⚠️ Metrics variables not found - first calculate them")






